# Session 2 — The Model That Cheated

## MISSION

> Four candidate models will be evaluated under a frozen chronological design. One candidate will be deliberately contaminated with future information, exposed, and disqualified before a legitimate champion is selected.

### Success criteria

- A prediction-time feature contract will be established.
- A chronological development–calibration–future-holdout structure will be created.
- Dummy, logistic, corrected nonlinear, and deliberately leaky candidates will be compared.
- Macro F1 will be calculated and interpreted correctly.
- The highest-scoring invalid candidate will be excluded **before** legitimate models are ranked.
- Three auditable artifacts will be exported.

**Primary artifact:** `Leakage-Aware Model Validator`


## MISSION MAP

`FUSED DATA → FREEZE TIME → UNDERSTAND MACRO F1 → TRAIN FAIR MODELS → BUILD THE CHEATING MODEL → EXCLUDE → SELECT`

### Central rule

> **Eligibility is decided before performance is ranked.**


## HOW TO USE THIS LIVE WORKSPACE

- **RUN** — execute the cell as provided.
- **FILL** — replace `...` with an implementation.
- **HINT** — open the collapsible hint only after an initial attempt.
- **CHECKPOINT** — execute the assertion to verify the result.
- **EVIDENCE** — interpret the output before continuing.

The workspace intentionally contains partial implementations. The objective is not merely to obtain output; the objective is to preserve a valid evaluation design.


## OPENING CHALLENGE

| Candidate | Macro F1 | Available at prediction time? | Initial status |
|---|---:|---|---|
| Logistic regression | 0.6357 | Yes | Eligible |
| Leaky XGBoost | 0.9654 | No — uses `future_loss_index` | Disqualified |

> The higher score will be rejected because the feature contract is violated.

**Prediction:** Which candidate would be selected by a naive `sort_values("macro_f1")` rule?


## F1 AND MACRO F1 — 90-SECOND FOUNDATION

**Precision** asks: of everything predicted as a class, how much was correct?

\[
Precision = 
rac{TP}{TP+FP}
\]

**Recall** asks: of all real members of that class, how much was found?

\[
Recall = 
rac{TP}{TP+FN}
\]

**F1** combines precision and recall through the harmonic mean:

\[
F1 = 
rac{2 	imes Precision 	imes Recall}{Precision + Recall}
\]

A weak precision or recall value cannot be hidden by a strong value on the other side.


## LAB ACTION 1 — CALCULATE ONE CLASS F1

A HIGH-risk class produced:

- `TP = 19`
- `FP = 6`
- `FN = 1`

Calculate precision, recall, and F1.

<details>
<summary><strong>Hint 1 — formulas</strong></summary>

Use:

- `precision = TP / (TP + FP)`
- `recall = TP / (TP + FN)`
- `f1 = 2 * precision * recall / (precision + recall)`

</details>

<details>
<summary><strong>Hint 2 — expected range</strong></summary>

Precision should be below recall because six false HIGH alarms were produced, while only one actual HIGH case was missed.

</details>


In [1]:
# The true-positive count for the HIGH-risk class is declared.
tp_high = 19

# The false-positive count for the HIGH-risk class is declared.
fp_high = 6

# The false-negative count for the HIGH-risk class is declared.
fn_high = 1

# FILL: Calculate the proportion of predicted HIGH cases that were genuinely HIGH.
precision_high = tp_high/(tp_high + fp_high)

# FILL: Calculate the proportion of actual HIGH cases that were detected.
recall_high = tp_high/(tp_high + fn_high)

# FILL: Combine precision and recall through the harmonic mean.
f1_high = 2*(precision_high * recall_high)/(precision_high + recall_high)

# The calculated evidence is displayed for inspection.
print(f"HIGH precision: {precision_high:.4f}")

# The calculated evidence is displayed for inspection.
print(f"HIGH recall:    {recall_high:.4f}")

# The calculated evidence is displayed for inspection.
print(f"HIGH F1:        {f1_high:.4f}")


HIGH precision: 0.7600
HIGH recall:    0.9500
HIGH F1:        0.8444


In [3]:
# CHECKPOINT: Precision should equal 19 divided by 25.
assert round(precision_high, 4) == 0.7600, "Recheck the precision denominator."

# CHECKPOINT: Recall should equal 19 divided by 20.
assert round(recall_high, 4) == 0.9500, "Recheck the recall denominator."

# CHECKPOINT: The resulting F1 should be approximately 0.8444.
assert round(f1_high, 4) == 0.8444, "Recheck the harmonic-mean equation."

# The successful checkpoint is reported.
print("F1 checkpoint passed.")


F1 checkpoint passed.


## WHY MACRO F1 IS USED

For three classes:

\[
Macro\ F1 = 
rac{F1_{LOW}+F1_{MEDIUM}+F1_{HIGH}}{3}
\]

Each class receives equal influence, regardless of its number of records.

> Accuracy gives every **record** equal weight. Macro F1 gives every **class** equal weight.

This is important because strong LOW-risk performance must not hide failure on a smaller but operationally critical HIGH-risk class.

### Critical limitation

> Macro F1 evaluates prediction behavior; it does **not** detect leakage. Feature eligibility and evaluation design must be checked separately.


## TECHNICAL INTEGRITY NOTE

This workspace uses 2025 as a **later-year selection holdout** to demonstrate temporal generalization. In a stricter research workflow, model selection would be completed before a separate untouched final test set is opened.

The practical lesson remains unchanged:

> Training must not use future records, and the selection process must not use features unavailable at prediction time.


## 1. RUN — Path setup and imports


In [4]:
# The system module is imported so Python's module search path can be updated.
import sys

# Path is imported so filesystem locations can be handled portably.
from pathlib import Path

# The current working directory is captured as the starting location.
current = Path.cwd()

# Every current or parent directory is inspected for the project source folder.
for parent in [current] + list(current.parents):
    # The project root is accepted when a src directory is present.
    if (parent / "src").exists():
        # The detected project root is stored.
        root = parent

        # The upward search is stopped after the first valid root is found.
        break

# The fallback branch is used when no parent contains src.
else:
    # The current directory is used as the project root fallback.
    root = current

# The source directory is added only when it is not already importable.
if str(root / "src") not in sys.path:
    # The source directory is inserted at the beginning of Python's search path.
    sys.path.insert(0, str(root / "src"))

# The canonical data directory is defined relative to the project root.
DATA_DIR = root / "src" / "data"

# The canonical reports directory is defined relative to the project root.
REPORTS_DIR = root / "src" / "reports"

# The output directory is created when it does not already exist.
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# The resolved project root is displayed.
print(f"Project root: {root}")

# The resolved data directory is displayed.
print(f"Data dir: {DATA_DIR}")

# The resolved reports directory is displayed.
print(f"Reports dir: {REPORTS_DIR}")


Project root: c:\Users\Metwalli\Desktop\summer\Advanced_AI_Practical
Data dir: c:\Users\Metwalli\Desktop\summer\Advanced_AI_Practical\src\data
Reports dir: c:\Users\Metwalli\Desktop\summer\Advanced_AI_Practical\src\reports


In [7]:
# NumPy is imported for arrays, class labels, and numerical operations.
import numpy as np

# Pandas is imported for tabular data loading, filtering, and export.
import pandas as pd

# A fixed random seed is declared so repeated runs remain reproducible.
RANDOM_STATE = 42

# The composite observation key is declared for region-month records.
KEYS = ["region_id", "year", "month"]

# The ordered multiclass risk labels are declared.
CLASS_NAMES = np.array(["LOW", "MEDIUM", "HIGH"])

# A deterministic mapping from class names to integer IDs is created.
LABEL_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}

# The approved raw measurement features are declared.
RAW_FEATURES = [
    "temperature_c",
    "rainfall_mm",
    "humidity_pct",
    "soil_moisture",
    "soil_ph",
    "vegetation_index",
    "disease_cases",
    "pest_reports",
    "irrigation_capacity",
    "vulnerability_index",
]

# The engineered modeling features are declared.
ENGINEERED_FEATURES = [
    "drought_index",
    "heat_stress",
    "disease_pressure",
    "water_balance",
    "season_sin",
    "season_cos",
]

# The complete legitimate feature contract is assembled.
MODEL_FEATURES = RAW_FEATURES + ENGINEERED_FEATURES

# Post-outcome variables are explicitly quarantined.
PROHIBITED_FEATURES = ["future_loss_index", "response_action_code"]

# The number of legitimate model features is displayed.
print(f"Model features: {len(MODEL_FEATURES)}")

# The prohibited feature registry is displayed.
print(f"Prohibited features: {PROHIBITED_FEATURES}")


Model features: 16
Prohibited features: ['future_loss_index', 'response_action_code']


## 2. RUN — Load the fused modeling table


In [8]:
# The processed-data directory is resolved.
processed_dir = DATA_DIR / "processed"

# The expected fused modeling-table path is constructed.
fused_path = processed_dir / "modeling_table.csv"

# The fused table is generated only when the processed file is absent.
if not fused_path.exists():
    # A clear status message is displayed.
    print("Processed data not found. Generating from source data...")

    # The reusable fusion function is imported.
    from reliable_ai.data import fuse_sources

    # The raw sources are fused into one modeling table.
    fused = fuse_sources(DATA_DIR)

    # The processed directory is created when required.
    processed_dir.mkdir(parents=True, exist_ok=True)

    # The fused table is persisted for reproducible reuse.
    fused.to_csv(fused_path, index=False)

# The existing prepared table is used when available.
else:
    # The fused modeling table is loaded from disk.
    fused = pd.read_csv(fused_path)

# The resulting table dimensions are displayed.
print(f"Fused data: {fused.shape[0]} rows × {fused.shape[1]} columns")


Fused data: 840 rows × 24 columns


## 3. RUN AND INSPECT — Understand the modeling table


In [9]:
# The temporal range is displayed.
print(f"Year range: {fused['year'].min()}–{fused['year'].max()}")

# The number of distinct regions is displayed.
print(f"Regions: {fused['region_id'].nunique()}") 

# A readable heading is displayed before the class counts.
print("\nRisk-label distribution:")

# The number of records in each risk class is displayed.
print(fused["risk_label"].value_counts().to_string())

# The prohibited columns physically present in the table are reported.
print(
    "\nProhibited features present:",
    [feature for feature in PROHIBITED_FEATURES if feature in fused.columns],
)


Year range: 2019–2025
Regions: 10

Risk-label distribution:
risk_label
LOW       471
MEDIUM    347
HIGH       22

Prohibited features present: ['future_loss_index', 'response_action_code']


## PREDICT BEFORE RUNNING

Complete the statement before continuing:

> The deliberately leaky model is expected to produce a __________ Macro F1 because __________.

**Do not write “because XGBoost is advanced.”** The explanation must refer to information availability.


### Record the hypothesis

**Hypothesis:**  
`........................................................................................................`


## 4. FILL — Freeze time


Create three chronological partitions:

- Development: years up to and including 2023
- Calibration: 2024
- Later-year holdout: 2025

<details>
<summary><strong>Hint 1 — Boolean filtering</strong></summary>

Filter the `year` column with `<=`, `==`, and `==`.

</details>

<details>
<summary><strong>Hint 2 — Defensive copying</strong></summary>

Finish every filtered dataframe with `.copy()` so later modifications remain isolated.

</details>


In [ ]:
train = fused[(fused['year']>=2019) & (fused['year']<=2023)].copy()

In [ ]:
# FILL: Select all records from 2019 through 2023 and create an independent copy.
train = fused[(fused['year']>=2019) & (fused['year']<=2023)].copy()

# FILL: Select the 2024 records reserved for later calibration work.
calibration = ...

# FILL: Select the 2025 records used as the later-year holdout.
test = ...

# The development partition size and temporal range are displayed.
print(f"Train: {len(train)} rows ({train['year'].min()}–{train['year'].max()})")

# The calibration partition size and year are displayed.
print(f"Calibration: {len(calibration)} rows ({calibration['year'].min()})")

# The holdout partition size and year are displayed.
print(f"Holdout: {len(test)} rows ({test['year'].min()})")


In [ ]:
# CHECKPOINT: The development partition must contain records.
assert len(train) > 0, "The development partition is empty."

# CHECKPOINT: The calibration partition must contain records.
assert len(calibration) > 0, "The calibration partition is empty."

# CHECKPOINT: The holdout partition must contain records.
assert len(test) > 0, "The later-year holdout is empty."

# CHECKPOINT: Development must end before calibration begins.
assert train["year"].max() < calibration["year"].min(), "Development overlaps calibration."

# CHECKPOINT: Calibration must end before the holdout begins.
assert calibration["year"].max() < test["year"].min(), "Calibration overlaps the holdout."

# The successful split is reported.
print("Chronological split verified.")


## 5. FILL — Encode the target


The string labels must be mapped as:

- `LOW → 0`
- `MEDIUM → 1`
- `HIGH → 2`

<details>
<summary><strong>Hint 1 — mapping object</strong></summary>

Use the existing `LABEL_TO_ID` dictionary.

</details>

<details>
<summary><strong>Hint 2 — Pandas operation</strong></summary>

A Series can be converted with `.map(...)`.

</details>

<details>
<summary><strong>Hint 3 — final type</strong></summary>

Convert the mapped values to a NumPy integer array.

</details>


In [ ]:
# A reusable label-encoding function is defined.
def encode_target(labels):
    # FILL: Map string labels through LABEL_TO_ID and return integer values.
    return ...

# The development labels are encoded.
y_train = encode_target(train["risk_label"])

# The holdout labels are encoded.
y_test = encode_target(test["risk_label"])

# The encoded development classes are displayed.
print(f"Encoded train classes: {np.unique(y_train)}")

# The encoded holdout classes are displayed.
print(f"Encoded holdout classes: {np.unique(y_test)}")


In [ ]:
# CHECKPOINT: All three encoded classes must be represented in development data.
assert set(np.unique(y_train)) == {0, 1, 2}, "Unexpected development classes."

# CHECKPOINT: All three encoded classes must be represented in the holdout.
assert set(np.unique(y_test)) == {0, 1, 2}, "Unexpected holdout classes."

# The successful encoding is reported.
print("Target encoding verified.")


In [ ]:
# Only legitimate model features are selected for development.
X_train = train[MODEL_FEATURES]

# The same legitimate feature contract is applied to the holdout.
X_test = test[MODEL_FEATURES]

# The resulting feature matrix dimensions are displayed.
print(f"X_train: {X_train.shape}")

# The resulting holdout feature matrix dimensions are displayed.
print(f"X_holdout: {X_test.shape}")


## 6. RUN — Import modeling components


In [ ]:
# The trivial baseline classifier is imported.
from sklearn.dummy import DummyClassifier

# The median imputer is imported.
from sklearn.impute import SimpleImputer

# The numerical standardizer is imported.
from sklearn.preprocessing import StandardScaler

# Logistic regression is imported as the serious linear baseline.
from sklearn.linear_model import LogisticRegression

# The column-aware preprocessing transformer is imported.
from sklearn.compose import ColumnTransformer

# The end-to-end pipeline container is imported.
from sklearn.pipeline import Pipeline

# The shared classification metrics are imported.
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
)

# A readiness message is displayed.
print("Modeling components imported.")


## 7. RUN — Candidate 1: Dummy baseline


In [ ]:
# A majority-class baseline is instantiated as the performance floor.
dummy = DummyClassifier(
    strategy="most_frequent",
    random_state=RANDOM_STATE,
)

# The dummy baseline is fitted on legitimate development features.
dummy.fit(X_train, y_train)

# A completion message is displayed.
print("Dummy baseline trained.")


## 8. FILL — Candidate 2: Leakage-safe logistic pipeline


Complete the legitimate logistic pipeline.

<details>
<summary><strong>Hint 1 — preprocessing order</strong></summary>

Missing values should be imputed before numerical scaling.

</details>

<details>
<summary><strong>Hint 2 — imputation strategy</strong></summary>

Use the training-set median.

</details>

<details>
<summary><strong>Hint 3 — class imbalance</strong></summary>

Use a class-weight option that automatically assigns larger weight to smaller classes.

</details>

<details>
<summary><strong>Why the pipeline matters</strong></summary>

The imputer and scaler must be fitted only on the development partition. Placing them inside `Pipeline` prevents holdout statistics from entering training.

</details>


In [ ]:
# A numerical preprocessing pipeline is started.
numeric_transformer = Pipeline(
    steps=[
        # FILL: Replace missing values using a robust training-derived statistic.
        ("imputer", SimpleImputer(strategy=...)),

        # FILL: Standardize numerical features inside the pipeline.
        ("scaler", ...),
    ]
)

# A column-aware transformer applies the numerical pipeline to legitimate features.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, MODEL_FEATURES),
    ],
    remainder="drop",
)

# The end-to-end logistic pipeline is started.
logistic = Pipeline(
    steps=[
        # The leakage-safe preprocessing stage is registered.
        ("preprocessor", preprocessor),

        # FILL: Configure a reproducible, class-balanced logistic classifier.
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight=...,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

# The complete pipeline is fitted only on development data.
logistic.fit(X_train, y_train)

# A completion message is displayed.
print("Logistic pipeline trained.")


In [ ]:
# CHECKPOINT: The fitted object must expose both preprocessing and classification stages.
assert set(logistic.named_steps) == {"preprocessor", "classifier"}, "Pipeline stages are incomplete."

# CHECKPOINT: The classifier must use class balancing.
assert logistic.named_steps["classifier"].class_weight == "balanced", "Class balancing is missing."

# The successful pipeline configuration is reported.
print("Logistic pipeline verified.")


## 9. RUN — Candidate 3: Correct nonlinear model


In [ ]:
# The preferred XGBoost implementation is attempted.
try:
    # The XGBoost multiclass classifier is imported.
    from xgboost import XGBClassifier

    # A nonlinear multiclass candidate is configured.
    nonlinear_classifier = XGBClassifier(
        n_estimators=260,
        max_depth=4,
        learning_rate=0.055,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=1,
        device="cpu",
    )

    # The active implementation is reported.
    print("Using XGBClassifier.")

# A compatible fallback is used when XGBoost is unavailable.
except ImportError:
    # The fallback gradient-boosting classifier is imported.
    from sklearn.ensemble import HistGradientBoostingClassifier

    # A compatible nonlinear candidate is configured.
    nonlinear_classifier = HistGradientBoostingClassifier(
        learning_rate=0.07,
        max_iter=220,
        max_leaf_nodes=15,
        l2_regularization=0.15,
        random_state=RANDOM_STATE,
    )

    # The active fallback is reported.
    print("Using HistGradientBoostingClassifier.")

# A legitimate nonlinear preprocessing pipeline is configured.
advanced_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            MODEL_FEATURES,
        ),
    ],
    remainder="drop",
)

# The legitimate nonlinear end-to-end pipeline is configured.
advanced_correct = Pipeline(
    steps=[
        ("preprocessor", advanced_preprocessor),
        ("classifier", nonlinear_classifier),
    ]
)

# The corrected nonlinear candidate is fitted on legitimate features only.
advanced_correct.fit(X_train, y_train)

# A completion message is displayed.
print("Correct nonlinear candidate trained.")


## 10. FILL — Candidate 4: Deliberately leaky model


The same nonlinear family will now be contaminated intentionally.

<details>
<summary><strong>Hint 1 — leakage feature</strong></summary>

Append exactly one post-outcome feature to `MODEL_FEATURES`.

</details>

<details>
<summary><strong>Hint 2 — feature name</strong></summary>

The prohibited feature is named `future_loss_index`.

</details>

<details>
<summary><strong>Interpretation</strong></summary>

This candidate is an experimental failure case. It is created to demonstrate how invalid information can inflate metrics.

</details>


In [ ]:
# FILL: Append the prohibited post-outcome proxy to the legitimate feature registry.
leaky_features = ...

# A preprocessing transformer is configured for the contaminated feature set.
leaky_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            leaky_features,
        ),
    ],
    remainder="drop",
)

# The preferred XGBoost implementation is attempted.
try:
    # The XGBoost multiclass classifier is imported.
    from xgboost import XGBClassifier

    # A second nonlinear classifier is configured for the controlled leakage experiment.
    leaky_classifier = XGBClassifier(
        n_estimators=260,
        max_depth=4,
        learning_rate=0.055,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=1,
        device="cpu",
    )

# A compatible fallback is used when XGBoost is unavailable.
except ImportError:
    # The fallback gradient-boosting classifier is imported.
    from sklearn.ensemble import HistGradientBoostingClassifier

    # A second fallback classifier is configured.
    leaky_classifier = HistGradientBoostingClassifier(
        learning_rate=0.07,
        max_iter=220,
        max_leaf_nodes=15,
        l2_regularization=0.15,
        random_state=RANDOM_STATE,
    )

# The contaminated end-to-end pipeline is configured.
leaky_pipeline = Pipeline(
    steps=[
        ("preprocessor", leaky_preprocessor),
        ("classifier", leaky_classifier),
    ]
)

# FILL: Fit the contaminated candidate using the development rows and leaky feature set.
leaky_pipeline.fit(..., y_train)

# A completion message is displayed.
print("Deliberately leaky candidate trained.")


In [ ]:
# CHECKPOINT: The deliberately contaminated feature set must contain future_loss_index.
assert "future_loss_index" in leaky_features, "The leakage feature was not added."

# CHECKPOINT: The legitimate feature set must remain uncontaminated.
assert "future_loss_index" not in MODEL_FEATURES, "The legitimate feature registry was modified."

# The successful controlled contamination is reported.
print("Controlled leakage experiment configured correctly.")


## 11. FILL — Define one shared metric contract


Every candidate must be evaluated by the same function.

<details>
<summary><strong>Hint 1 — accuracy</strong></summary>

Use `accuracy_score(y_true, y_pred)`.

</details>

<details>
<summary><strong>Hint 2 — balanced accuracy</strong></summary>

Use `balanced_accuracy_score(y_true, y_pred)`.

</details>

<details>
<summary><strong>Hint 3 — Macro F1</strong></summary>

Use `f1_score(..., average="macro")`.

</details>

<details>
<summary><strong>Hint 4 — probability quality</strong></summary>

Use `log_loss(y_true, probabilities)`.

</details>


In [ ]:
# A reusable candidate-evaluation function is defined.
def metric_row(name, model, data, y_true, features):
    # Class labels are predicted from the candidate-specific feature set.
    y_pred = model.predict(data[features])

    # Class probabilities are generated for probability-sensitive evaluation.
    probabilities = model.predict_proba(data[features])

    # One structured metric record is returned.
    return {
        # The candidate name is recorded.
        "model": name,

        # FILL: Calculate overall classification accuracy.
        "accuracy": ...,

        # FILL: Calculate recall averaged equally across classes.
        "balanced_accuracy": ...,

        # FILL: Calculate per-class F1 and average all classes equally.
        "macro_f1": ...,

        # FILL: Measure the quality of the predicted class probabilities.
        "log_loss": ...,
    }

# A readiness message is displayed.
print("Shared metric contract defined.")


## 12. RUN — Score all four candidates


In [ ]:
# Every candidate and its required feature contract is registered.
models = {
    "dummy": (dummy, MODEL_FEATURES),
    "logistic": (logistic, MODEL_FEATURES),
    "advanced_correct": (advanced_correct, MODEL_FEATURES),
    "advanced_leaky": (leaky_pipeline, leaky_features),
}

# An empty list is created for candidate metric records.
rows = []

# Every registered candidate is evaluated using the shared metric contract.
for name, (model, features) in models.items():
    # One candidate metric row is calculated.
    row = metric_row(name, model, test, y_test, features)

    # The metric row is appended to the comparison collection.
    rows.append(row)

# The metric records are converted into one comparison dataframe.
metrics = pd.DataFrame(rows)

# A completion message is displayed.
print("All candidates scored.")


In [ ]:
# All candidates are naively ordered by Macro F1 before eligibility is considered.
all_sorted = metrics.sort_values(
    by="macro_f1",
    ascending=False,
).reset_index(drop=True)

# The naive raw ranking is displayed.
print("=== RAW RANKING — ALL CANDIDATES ===")

# The complete metric table is displayed.
print(all_sorted.to_string(index=False))


## PREDICTION CHECKPOINT

Before the next cell:

1. Which model should appear first in the raw ranking?
2. Which model should be selected after eligibility filtering?
3. Why are those answers different?

Record the answer before continuing.


### Record the prediction

**Raw winner:** `........................................`

**Legitimate winner:** `........................................`

**Reason:** `........................................................................................................`


## 13. BREAK IT — Select by score alone


In [ ]:
# The raw metric table is sorted without any feature-eligibility rule.
naive_best = metrics.sort_values(
    by="macro_f1",
    ascending=False,
).iloc[0]

# The naive winner is displayed.
print(f"Naive winner: {naive_best['model']}")

# The naive winning Macro F1 is displayed.
print(f"Naive Macro F1: {naive_best['macro_f1']:.4f}")

# The scientific failure is stated explicitly.
print("Failure: score-only selection ignores prediction-time feature availability.")


> A metric function cannot know that a feature came from the future.  
> The model-selection system must enforce that contract.


## 14. FILL — Exclude invalid candidates before ranking


<details>
<summary><strong>Hint 1 — legitimate names</strong></summary>

Use: `dummy`, `logistic`, and `advanced_correct`.

</details>

<details>
<summary><strong>Hint 2 — Pandas filtering</strong></summary>

Filter `metrics["model"]` with `.isin(...)`.

</details>

<details>
<summary><strong>Hint 3 — isolation</strong></summary>

Finish the filtered dataframe with `.copy()`.

</details>


In [ ]:
# The approved candidate names are declared before ranking.
LEGITIMATE_MODEL_NAMES = [
    "dummy",
    "logistic",
    "advanced_correct",
]

# FILL: Keep only rows whose model name belongs to the approved registry.
legitimate = ...

# The surviving candidate names are displayed.
print(f"Legitimate candidates: {legitimate['model'].tolist()}")


In [ ]:
# CHECKPOINT: Exactly three legitimate candidates must survive.
assert len(legitimate) == 3, "The eligibility filter returned the wrong number of candidates."

# CHECKPOINT: The deliberately leaky candidate must be absent.
assert "advanced_leaky" not in legitimate["model"].tolist(), "The leaky candidate was not excluded."

# The successful eligibility filter is reported.
print("Eligibility filter verified.")


## 15. FILL — Apply the evidence-based selection rule


Selection rule:

1. Highest legitimate Macro F1
2. Lowest log loss when Macro F1 ties
3. Alphabetical model name as a deterministic final tie-breaker

<details>
<summary><strong>Hint</strong></summary>

Use `sort_values` with:

- columns: `["macro_f1", "log_loss", "model"]`
- ascending flags: `[False, True, True]`

Then restore a clean index and take row zero.

</details>


In [ ]:
# FILL: Rank legitimate candidates by the declared multi-column rule.
sorted_legitimate = ...

# The legitimate ranking is displayed.
print("=== LEGITIMATE RANKING ===")

# The legitimate comparison table is displayed.
print(sorted_legitimate.to_string(index=False))

# FILL: Select the first row from the legitimate ranking.
selected = ...

# The selected model name is displayed.
print(f"Selected model: {selected['model']}")

# The selected Macro F1 is displayed.
print(f"Selected Macro F1: {selected['macro_f1']:.4f}")

# The selected log loss is displayed.
print(f"Selected log loss: {selected['log_loss']:.4f}")


In [ ]:
# CHECKPOINT: The deterministic package is expected to select logistic regression.
assert selected["model"] == "logistic", f"Expected logistic, received {selected['model']}."

# The successful selection is reported.
print("Evidence-based selection verified.")


## 16. FILL — Export the audit artifacts


In [ ]:
# The full four-candidate comparison is exported.
metrics.to_csv(
    REPORTS_DIR / "model_comparison.csv",
    index=False,
)

# A completion message is displayed.
print("Exported model_comparison.csv.")


### Feature-availability audit hints

For legitimate features:

- `model_role = "approved"`
- `prediction_time_availability = "available"`
- `eligibility = "eligible"`

For prohibited features:

- `model_role = "prohibited"`
- `prediction_time_availability = "unavailable"`
- `eligibility = "ineligible"`
- reason should explain that the value is post-outcome or unavailable at prediction time.


In [ ]:
# An empty collection is created for feature-availability evidence.
audit_rows = []

# Every legitimate feature is added to the audit.
for feature in MODEL_FEATURES:
    # One eligible feature record is appended.
    audit_rows.append(
        {
            "feature": feature,
            "model_role": "approved",
            "prediction_time_availability": "available",
            "eligibility": "eligible",
            "reason": "available from current or historical measurements",
        }
    )

# Every quarantined feature is added to the audit.
for feature in PROHIBITED_FEATURES:
    # FILL: Append one ineligible feature record with an explicit reason.
    audit_rows.append(
        {
            "feature": feature,
            "model_role": ...,
            "prediction_time_availability": ...,
            "eligibility": ...,
            "reason": ...,
        }
    )

# The audit records are converted into a dataframe.
feature_audit = pd.DataFrame(audit_rows)

# The feature audit is exported.
feature_audit.to_csv(
    REPORTS_DIR / "feature_availability_audit.csv",
    index=False,
)

# The prohibited portion of the audit is displayed.
print(
    feature_audit[
        feature_audit["feature"].isin(PROHIBITED_FEATURES)
    ].to_string(index=False)
)


### Model-selection record hints

The record must preserve:

- selected model name and Macro F1;
- explicit selection rule;
- excluded candidate and exclusion reason;
- chronological partition contract.


In [ ]:
# The selected candidate metric row is isolated.
selected_row = metrics[
    metrics["model"] == selected["model"]
].iloc[0]

# The leaky candidate metric row is isolated for the audit trail.
leaky_row = metrics[
    metrics["model"] == "advanced_leaky"
].iloc[0]

# One auditable selection record is constructed.
selection_record = pd.DataFrame(
    [
        {
            # FILL: Record the legitimate selected model.
            "selected_model_name": ...,

            # FILL: Record the selected model's Macro F1.
            "selected_macro_f1": ...,

            # FILL: State the exact ordering rule.
            "selection_rule": ...,

            # FILL: Record the disqualified candidate.
            "excluded_candidate": ...,

            # FILL: State why the excluded model cannot be used in deployment.
            "excluded_reason": ...,

            # The development period is recorded.
            "development_years": "2019-2023",

            # The calibration period is recorded.
            "calibration_year": 2024,

            # The future selection holdout is recorded.
            "future_holdout_year": 2025,
        }
    ]
)

# The model-selection record is exported.
selection_record.to_csv(
    REPORTS_DIR / "model_selection_record.csv",
    index=False,
)

# The complete selection record is displayed.
print(selection_record.to_string(index=False))


## 17. EVIDENCE — Leakage-Aware Model Validator


In [ ]:
# A visual report separator is displayed.
print("=" * 66)

# The validator title is displayed.
print("LEAKAGE-AWARE MODEL VALIDATOR")

# A visual report separator is displayed.
print("=" * 66)

# The total candidate count is displayed.
print(f"{'Total candidates':32s}: {len(metrics)}")

# The legitimate candidate count is displayed.
print(f"{'Eligible candidates':32s}: {len(legitimate)}")

# The excluded model name is displayed.
print(f"{'Excluded candidate':32s}: advanced_leaky")

# The excluded model's Macro F1 is displayed.
print(f"{'Leaky Macro F1':32s}: {leaky_row['macro_f1']:.4f}")

# The selected model name is displayed.
print(f"{'Selected model':32s}: {selected['model']}")

# The selected model's Macro F1 is displayed.
print(f"{'Selected Macro F1':32s}: {selected['macro_f1']:.4f}")

# The scientific selection rule is displayed.
print(f"{'Selection rule':32s}: eligibility first, then Macro F1, then log loss")

# The successful validation status is displayed.
print(f"{'Validation status':32s}: PASSED")


## EVIDENCE TO RECORD

Answer before submission:

1. Why can an invalid model obtain the highest Macro F1?
2. Why does Macro F1 not detect future-feature leakage?
3. Why must imputation and scaling remain inside the pipeline?
4. Why does logistic regression win in this deterministic run?
5. What evidence would justify selecting XGBoost on a different dataset?
6. Which class appears most difficult, and how was that conclusion reached?


## ASSIGNMENT 02 — BUILD A MODEL THAT CHEATS, THEN CATCH IT

Continue the independent fused dataset from Assignment 01.

Required outputs:

- prediction-time contract;
- justified chronological or group-aware split;
- dummy baseline;
- serious linear baseline;
- legitimate nonlinear candidate;
- deliberately suspicious candidate;
- feature-availability audit;
- accuracy, balanced accuracy, Macro F1, and log loss;
- eligibility-first model-selection record;
- selected-model confusion matrix and technical interpretation.

**Submission principle:** Code without evidence and interpretation is incomplete.


## SESSION RESULT

> A later-year model-selection process has been constructed, a deliberately inflated model has been exposed, and the strongest legitimate candidate has been selected through an auditable rule.

The next session opens the selected model and investigates what drives its decisions.


## TECHNICAL BASIS

- F1 and Macro F1: scikit-learn `f1_score`
- Leakage-safe preprocessing: scikit-learn common pitfalls and pipelines
- Chronological validation: scikit-learn `TimeSeriesSplit`

Reference links:

- https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html
- https://scikit-learn.org/stable/common_pitfalls.html
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html
